In [4]:
import rapidsegment as rs

In [5]:
rs.__version__

'1.1.4'

In [6]:
from rapidsegment import StrategicSegmentBuilder, StrategicSegmentScore, UniversalDataLoader


In [7]:
data = UniversalDataLoader(file_path = r"/workspaces/RapidSegment/student_burnout_dropout_dataset_2.csv").load()

In [8]:
import pyarrow as pa

# Dictionary comprehension to get counts for all columns
null_counts = {name: data[name].null_count for name in data.column_names}
print(null_counts)
# Output example: {'col1': 0, 'col2': 3, 'col3': 1}


{'Student_ID': 0, 'Age': 0, 'Gender': 0, 'Year_of_Study': 0, 'Department': 0, 'Residence_Type': 0, 'Attendance_Percent': 51, 'Study_Hours_Per_Day': 35, 'Previous_GPA': 35, 'Backlogs': 15, 'Sleep_Hours': 35, 'Screen_Time_Hours': 36, 'Exercise_Freq_Per_Week': 0, 'Social_Activity_Score': 40, 'Part_Time_Job': 0, 'Family_Income_Bracket': 0, 'Financial_Stress_Score': 43, 'Family_Support_Score': 44, 'Stress_Level': 49, 'Anxiety_Score': 39, 'Motivation_Score': 41, 'Peer_Pressure_Score': 41, 'Counseling_Access': 0, 'Burnout_Level': 0, 'Dropout_Risk': 0}


In [9]:
print(data.schema)


Student_ID: string
Age: double
Gender: string
Year_of_Study: double
Department: string
Residence_Type: string
Attendance_Percent: double
Study_Hours_Per_Day: double
Previous_GPA: double
Backlogs: double
Sleep_Hours: double
Screen_Time_Hours: double
Exercise_Freq_Per_Week: double
Social_Activity_Score: double
Part_Time_Job: string
Family_Income_Bracket: string
Financial_Stress_Score: double
Family_Support_Score: double
Stress_Level: double
Anxiety_Score: double
Motivation_Score: double
Peer_Pressure_Score: double
Counseling_Access: string
Burnout_Level: string
Dropout_Risk: string


In [10]:

import pyarrow.compute as pc


# 2. Define your replacements
condition = pc.equal(data["Dropout_Risk"], "Yes")
new_values = pc.if_else (condition, 1, 0)

# 3. Replace the old column with the new computed array in the immutable table
new_table = data.set_column(
    data.schema.get_field_index("Dropout_Risk"), 
    "Dropout_Risk", 
    new_values
)


In [11]:
grid_config = {
    "min_sample_size": [100, 50],
    "min_lift": [2.0, 1.2]
}


In [12]:
builder = StrategicSegmentBuilder(
    target="Dropout_Risk",
    min_sample_size=50,
    min_lift=1.2,
    min_events=5,
    top_n_vars=10,
    max_segments=10,
    max_feature_reuse=1,
    param_grid=None,
    enable_diversity=True,
    feature_groups=None,
    ignore_features=["Student_ID"]
)

In [13]:
segments_summary = builder.extract_segments(new_table)

2026-07-16 16:59:47,824 | INFO     | [builder.py:354] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-16 16:59:47,841 | INFO     | [builder.py:390] | Iteration 1 | Remaining Volume: 800 | Base Rate: 54.62%


2026-07-16 16:59:48,821 | INFO     | [builder.py:539] | Feature Usage Tracker Update -> 'Burnout_Level' used count = 1
2026-07-16 16:59:48,822 | INFO     | [builder.py:539] | Feature Usage Tracker Update -> 'Family_Income_Bracket' used count = 1
2026-07-16 16:59:48,822 | INFO     | [builder.py:539] | Feature Usage Tracker Update -> 'Study_Hours_Per_Day' used count = 1
2026-07-16 16:59:48,823 | INFO     | [builder.py:554] | Segment 1 Captured (Size Floor: 50 | Lift Floor: 1.2): Burnout_Level IN ('High') AND Family_Income_Bracket IN ('Lower-Middle') AND (Study_Hours_Per_Day >= 0.05 AND Study_Hours_Per_Day < 3.45)
2026-07-16 16:59:48,833 | INFO     | [builder.py:390] | Iteration 2 | Remaining Volume: 747 | Base Rate: 51.67%
2026-07-16 16:59:49,968 | INFO     | [builder.py:539] | Feature Usage Tracker Update -> 'Attendance_Percent' used count = 1
2026-07-16 16:59:49,969 | INFO     | [builder.py:554] | Segment 2 Captured (Size Floor: 50 | Lift Floor: 1.2): Attendance_Percent < 52.45
2026-07

In [14]:
import pandas as pd
segments_df = pd.DataFrame(segments_summary)
print("\nExtracted Segment Profiles:")
segments_df[["segment_id", "count", "lift", "meta_applied_sample_size", "sql_filter"]]



Extracted Segment Profiles:


,segment_id,count,lift,meta_applied_sample_size,sql_filter
0,1,51,1.758873,50,Burnout_Level IN ('High') AND Family_Income_Br...
1,2,90,1.333161,50,Attendance_Percent < 52.45
2,3,93,1.286440,50,Stress_Level >= 7.45


In [16]:
coverage_report = builder.evaluate_final_coverage(new_table)
print("\nCascading Portfolio Coverage Analysis:")
pd.DataFrame(coverage_report)



Cascading Portfolio Coverage Analysis:


,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,566,267.0,47.173145,54.625,70.750,0.863582
1,1,51,49.0,96.078431,54.625,6.375,1.758873
2,2,90,62.0,68.888889,54.625,11.250,1.261124
3,3,93,59.0,63.440860,54.625,11.625,1.161389


In [ ]:
import duckdb
con = duckdb.connect(database='temp_scoring.db', read_only=False)

In [ ]:
con.execute("CREATE OR REPLACE TABLE scored_data AS SELECT * FROM new_table")

In [ ]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   Family_Income_Bracket=<ArrowStringArray>
['Lower-Middle']
Length: 1, dtype: str & Burnout_Level=<ArrowStringArray>
['High']
Length: 1, dtype: str & Study_Hours_Per_Day=[0.05, 3.45)
SQL Filter: Family_Income_Bracket IN ('Lower-Middle') AND Burnout_Level IN ('High') AND (Study_Hours_Per_Day >= 0.05 AND Study_Hours_Per_Day < 3.45)
--------------------------------------------------
Segment ID: 2
Raw Rule:   Attendance_Percent=(-inf, 52.45)
SQL Filter: Attendance_Percent < 52.45
--------------------------------------------------
Segment ID: 3
Raw Rule:   Stress_Level=[7.45, inf)
SQL Filter: Stress_Level >= 7.45
--------------------------------------------------


In [ ]:
con.execute("""SELECT CASE WHEN 
            (COALESCE(Study_Hours_Per_Day, 0) >= 0.05 AND COALESCE(Study_Hours_Per_Day, 0) < 3.45) AND Burnout_Level IN ('High') AND Family_Income_Bracket IN ('Lower-Middle') THEN 1 ELSE 0 END AS SEG_1,
CASE WHEN  COALESCE(Attendance_Percent,0) < 52.45 THEN 1 ELSE 0 END AS SEG_2,
CASE WHEN COALESCE(Stress_Level, 0) >= 7.45 THEN 1 ELSE 0 END AS SEG_3,
COUNT(*) AS count,
SUM(Dropout_Risk) AS events
FROM scored_data
GROUP BY ALL""").fetchdf()

,SEG_1,SEG_2,SEG_3,count,events
0,0,0,0,566,267.0
1,0,0,1,93,59.0
2,0,1,0,76,50.0
3,0,1,1,14,12.0
4,1,0,0,35,34.0
5,1,0,1,14,13.0
6,1,1,0,1,1.0
7,1,1,1,1,1.0


In [ ]:
con.close()